In [2]:
#!pip install sentence-transformers langdetect deep-translator

In [8]:
import re
import pandas as pd
from deep_translator import GoogleTranslator
from langdetect import DetectorFactory, detect
from sentence_transformers import SentenceTransformer, util

DetectorFactory.seed = 0          # same language detection on every run

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 3                         # how many laptops to show
MIN_SCORE = 0.25                  # minimum similarity for a meaning-only search
DEBUG = True                      # print what the bot understood

INTRO = "Here are the best matches:"
NO_RESULT = "Sorry, I couldn't find a matching laptop in the data."

SORTS = {                         # sort name -> (columns, ascending?)
    "price_asc": (["price"], [True]),
    "price_desc": (["price"], [False]),
    "rating_desc": (["rating", "reviews"], [False, False]),
    "reviews_desc": (["reviews"], [False]),
}

In [9]:
#Load data, model and embeddings

df = pd.read_csv("data/laptops_clean.csv")
df["text"] = df["name"] + ". " + df["description"]

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True)
print(df.shape, embeddings.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2998.98it/s]


(117, 10) torch.Size([117, 384])


In [8]:
#!pip install lingua-language-detector

   ---------------------------------------- 0.0/95.9 MB ? eta -:--:--
   - -------------------------------------- 3.1/95.9 MB 16.8 MB/s eta 0:00:06
   - -------------------------------------- 4.5/95.9 MB 11.2 MB/s eta 0:00:09
   ---- ----------------------------------- 9.7/95.9 MB 16.3 MB/s eta 0:00:06
   ------ --------------------------------- 14.9/95.9 MB 18.8 MB/s eta 0:00:05
   -------- ------------------------------- 21.5/95.9 MB 21.6 MB/s eta 0:00:04
   ------------ --------------------------- 29.4/95.9 MB 24.2 MB/s eta 0:00:03
   --------------- ------------------------ 37.5/95.9 MB 26.5 MB/s eta 0:00:03
   ------------------- -------------------- 46.4/95.9 MB 28.4 MB/s eta 0:00:02
   ----------------------- ---------------- 55.3/95.9 MB 29.9 MB/s eta 0:00:02
   --------------------------- ------------ 65.5/95.9 MB 31.9 MB/s eta 0:00:01
   ------------------------------- -------- 75.2/95.9 MB 33.3 MB/s eta 0:00:01
   ----------------------------------- ---- 85.7/95.9 MB 34.4 MB

In [12]:
#Language helpers

from lingua import Language, LanguageDetectorBuilder

# the languages the bot supports: add or remove lines to change them
detector = LanguageDetectorBuilder.from_languages(
    Language.ENGLISH, Language.FRENCH, Language.ARABIC
).build()


def detect_lang(text):
    lang = detector.detect_language_of(text)          # None if it can't decide
    return lang.iso_code_639_1.name.lower() if lang else "en"


def translate(text, target, source="auto"):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception:
        return text


tests = ["cheapest laptop", "laptops under 500", "a laptop for travel",
         "quel est l'ordinateur le moins cher", "ordinateur portable pas cher",
         "أرخص حاسوب", "ما هو أفضل حاسوب محمول"]
for t in tests:
    print(detect_lang(t), "|", t)

en | cheapest laptop
en | laptops under 500
en | a laptop for travel
fr | quel est l'ordinateur le moins cher
fr | ordinateur portable pas cher
ar | أرخص حاسوب
ar | ما هو أفضل حاسوب محمول


In [11]:
#!pip install sentencepiece sacremoses

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 17.8 MB/s  0:00:00
   ---------------------------------------- 0.0/867.8 kB ? eta -:--:--
   ---------------------------------------- 867.8/867.8 kB 19.5 MB/s  0:00:00

   ---------------------------------------- 2/2 [sacremoses]



In [4]:
from transformers import MarianMTModel, MarianTokenizer

_models = {}   # loaded on first use, then kept in memory

def _load(lang):
    if lang not in _models:
        name = f"Helsinki-NLP/opus-mt-{lang}-en"
        _models[lang] = (MarianTokenizer.from_pretrained(name), MarianMTModel.from_pretrained(name))
    return _models[lang]


def to_english(text, lang):
    """Translate a question from 'fr' or 'ar' to English, offline. On any error, return the text unchanged."""
    if lang == "en":
        return text
    try:
        tokenizer, mt_model = _load(lang)
        batch = tokenizer([text], return_tensors="pt", padding=True)
        output = mt_model.generate(**batch, max_new_tokens=64)
        return tokenizer.decode(output[0], skip_special_tokens=True)
    except Exception as e:
        print("   [translation error]", e)
        return text


for t in ["quel est l'ordinateur le moins cher", "meilleur ordinateur le mieux noté", "أرخص حاسوب"]:
    lang = detect_lang(t)
    print(lang, "|", t, "->", to_english(t, lang))

Loading weights: 100%|██████████| 256/256 [00:00<00:00, 897.74it/s]
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


fr | quel est l'ordinateur le moins cher -> which computer is the cheapest
fr | meilleur ordinateur le mieux noté -> best rated computer


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 3025.11it/s]
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ar | أرخص حاسوب -> The cheapest computer.


In [5]:
import re

def parse_query(question_en):
    q = question_en.lower()
    f = {"max_price": None, "min_ram": None, "sort": None}

    m = re.search(
        r"(?:under|below|less than|cheaper than|at most|up to|max(?:imum)?|budget(?: of)?)"
        r"\D{0,12}?(\d+(?:\.\d+)?)(?![\d.]*\s*(?:gb|tb|inch|\"))",
        q,
    )
    if m:
        f["max_price"] = float(m.group(1))

    m = re.search(r"(\d+)\s*gb\s*(?:of\s*)?ram", q) or re.search(r"ram\D{0,12}?(\d+)\s*gb", q)
    if m:
        f["min_ram"] = int(m.group(1))

    if re.search(r"cheapest|least expensive|lowest price|most affordable|\bcheap\b", q):
        f["sort"] = "price_asc"
    elif re.search(r"most expensive|priciest|highest price", q):
        f["sort"] = "price_desc"
    elif re.search(r"best[- ]rated|top[- ]rated|highest[- ]rated|highest rating|best rating", q):
        f["sort"] = "rating_desc"
    elif re.search(r"most reviewed|most popular", q):
        f["sort"] = "reviews_desc"
    return f


def apply_filters(frame, f):
    out = frame
    if f["max_price"] is not None:
        out = out[out["price"] <= f["max_price"]]
    if f["min_ram"] is not None:
        out = out[out["ram_gb"] >= f["min_ram"]]
    return out


for q in ["laptop under 500",
          "cheapest laptop with 8GB RAM",
          "which computer is the cheapest",      # translated from French
          "best rated computer",                 # translated from French
          "a laptop for travel"]:
    print(q, "->", parse_query(q))

laptop under 500 -> {'max_price': 500.0, 'min_ram': None, 'sort': None}
cheapest laptop with 8GB RAM -> {'max_price': None, 'min_ram': 8, 'sort': 'price_asc'}
which computer is the cheapest -> {'max_price': None, 'min_ram': None, 'sort': 'price_asc'}
best rated computer -> {'max_price': None, 'min_ram': None, 'sort': 'rating_desc'}
a laptop for travel -> {'max_price': None, 'min_ram': None, 'sort': None}


In [10]:
def search(question, question_en, df, model, embeddings):
    f = parse_query(question_en)
    if DEBUG:
        print(f"   [debug] English: {question_en!r} | rules found: {f}")

    # similarity between the question and every laptop (the model is multilingual)
    q_emb = model.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, embeddings)[0].cpu().numpy()
    scored = df.assign(score=scores)

    candidates = apply_filters(scored, f)
    if candidates.empty:
        return candidates

    if f["sort"]:
        cols, ascending = SORTS[f["sort"]]
        return candidates.sort_values(cols, ascending=ascending).head(TOP_K)

    candidates = candidates.sort_values("score", ascending=False).head(TOP_K)
    has_filter = f["max_price"] is not None or f["min_ram"] is not None
    if not has_filter:
        candidates = candidates[candidates["score"] >= MIN_SCORE]   # refuse weak matches
    return candidates


# quick test: (original question, its English version)
tests = [
    ("cheapest laptop with 8GB RAM", "cheapest laptop with 8GB RAM"),
    ("laptop under 500", "laptop under 500"),
    ("quel est l'ordinateur le moins cher", "which computer is the cheapest"),
    ("a laptop for travel", "a laptop for travel"),
    ("pizza recipe", "pizza recipe"),
]
for q, q_en in tests:
    print("Q:", q)
    display(search(q, q_en, df, model, embeddings)[["name", "price", "ram_gb", "rating", "score"]])

Q: cheapest laptop with 8GB RAM
   [debug] English: 'cheapest laptop with 8GB RAM' | rules found: {'max_price': None, 'min_ram': 8, 'sort': 'price_asc'}


,name,price,ram_gb,rating,score
43,Lenovo V110-15ISK,409.63,8.0,3,0.595095
5,Aspire E1-572G,581.99,8.0,1,0.457835
17,Acer Aspire A515-51-5654,679.00,8.0,2,0.494547


Q: laptop under 500
   [debug] English: 'laptop under 500' | rules found: {'max_price': 500.0, 'min_ram': None, 'sort': None}


,name,price,ram_gb,rating,score
31,Prestigio SmartBook 133S Gold,299.00,4.0,4,0.594275
1,Aspire E1-510,306.99,4.0,3,0.589361
46,Asus EeeBook R416NA-FA014T,433.30,4.0,1,0.583889


Q: quel est l'ordinateur le moins cher
   [debug] English: 'which computer is the cheapest' | rules found: {'max_price': None, 'min_ram': None, 'sort': 'price_asc'}


,name,price,ram_gb,rating,score
29,Asus VivoBook X441NA-GA190,295.99,4.0,3,0.477065
30,Prestigio SmartBook 133S Dark Grey,299.00,4.0,2,0.517798
31,Prestigio SmartBook 133S Gold,299.00,4.0,4,0.545834


Q: a laptop for travel
   [debug] English: 'a laptop for travel' | rules found: {'max_price': None, 'min_ram': None, 'sort': None}


,name,price,ram_gb,rating,score
85,Acer TravelMate P645-S-511A Black,1170.10,8.0,1,0.518532
75,Dell Latitude 5480,1133.82,8.0,4,0.484971
69,Dell Latitude 5480,1110.14,8.0,3,0.484331


Q: pizza recipe
   [debug] English: 'pizza recipe' | rules found: {'max_price': None, 'min_ram': None, 'sort': None}


,name,price,ram_gb,rating,score


In [13]:
MESSAGES = {
    "en": {"intro": "Here are the best matches:",
           "none": "Sorry, I couldn't find a matching laptop in the data."},
    "fr": {"intro": "Voici les meilleurs résultats :",
           "none": "Désolé, je n'ai trouvé aucun ordinateur correspondant dans les données."},
    "ar": {"intro": "هذه أفضل النتائج:",
           "none": "عذرًا، لم أجد حاسوبًا مطابقًا في البيانات."},
}

In [14]:
def format_product(row):
    return (
        f"- {row['name']} | ${row['price']:.2f} | rating {row['rating']}/5 "
        f"({row['reviews']} reviews)\n  {row['description']}"
    )


def answer(user_input, df, model, embeddings):
    lang = detect_lang(user_input)
    question_en = to_english(user_input, lang)
    if DEBUG:
        print(f"   [debug] detected language: {lang}")

    msgs = MESSAGES.get(lang, MESSAGES["en"])
    results = search(user_input, question_en, df, model, embeddings)
    if results.empty:
        return msgs["none"]

    lines = [format_product(row) for _, row in results.iterrows()]
    return msgs["intro"] + "\n" + "\n".join(lines)


for q in ["cheapest laptop with 8GB RAM",
          "quel est l'ordinateur le moins cher",
          "أرخص حاسوب",
          "a laptop for travel",
          "pizza recipe"]:
    print("Q:", q)
    print(answer(q, df, model, embeddings))
    print("-" * 60)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: cheapest laptop with 8GB RAM
   [debug] detected language: en
   [debug] English: 'cheapest laptop with 8GB RAM' | rules found: {'max_price': None, 'min_ram': 8, 'sort': 'price_asc'}
Here are the best matches:
- Lenovo V110-15ISK | $409.63 | rating 3/5 (9 reviews)
  Lenovo V110-15ISK, 15.6" HD, Core i3-6006U, 8GB, 128GB SSD, Windows 10 Home
- Aspire E1-572G | $581.99 | rating 1/5 (2 reviews)
  15.6", Core i5-4200U, 8GB, 1TB, Radeon R7 M265, Windows 8.1
- Acer Aspire A515-51-5654 | $679.00 | rating 2/5 (9 reviews)
  Acer Aspire A515-51-5654, Black, 15.6", FHD, Core i5-8250U, 8GB DDR4, 256GB SSD, Windows 10 Home, ENG
------------------------------------------------------------
Q: quel est l'ordinateur le moins cher


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [debug] detected language: fr
   [debug] English: 'which computer is the cheapest' | rules found: {'max_price': None, 'min_ram': None, 'sort': 'price_asc'}
Voici les meilleurs résultats :
- Asus VivoBook X441NA-GA190 | $295.99 | rating 3/5 (14 reviews)
  Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd
- Prestigio SmartBook 133S Dark Grey | $299.00 | rating 2/5 (8 reviews)
  Prestigio SmartBook 133S Dark Grey, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam
- Prestigio SmartBook 133S Gold | $299.00 | rating 4/5 (12 reviews)
  Prestigio SmartBook 133S Gold, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam
------------------------------------------------------------
Q: أرخص حاسوب
   [debug] detected language: ar
   [debug] English: 'The cheapest computer.' | rules found: {'max_price': None, 'min_ram': None, 'sort': 'price_asc'}
هذه أفضل النتائج:
- Asus VivoBook X44

In [15]:
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("exit", "quit"):
        print("Bye!")
        break
    if not user_input:
        continue
    print(f"\nBot: {answer(user_input, df, model, embeddings)}\n")

You:  best laptop


   [debug] detected language: en
   [debug] English: 'best laptop' | rules found: {'max_price': None, 'min_ram': None, 'sort': None}

Bot: Here are the best matches:
- ProBook | $739.99 | rating 4/5 (8 reviews)
  14", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit
- Lenovo ThinkPad L570 | $999.00 | rating 3/5 (11 reviews)
  Lenovo ThinkPad L570, 15.6" FHD, Core i7-7500U, 8GB, 256GB SSD, Windows 10 Pro
- Lenovo V110-15IKB | $465.95 | rating 1/5 (7 reviews)
  Lenovo V110-15IKB, 15.6" HD, Core i5-7200U, 4GB, 500GB, DOS



You:  exit


Bye!
